# Day 9: Data Cleaning & Preparation

In this notebook, I'll clean a messy real-world transaction dataset. Real-world datasets often have missing values, duplicate entries, inconsistent formats, and wrong data types. 

I will go through:
1. Finding and removing duplicates (exact and key-based).
2. Standardizing text columns (segment names, categories, and country formats).
3. Fixing inconsistent date formats.
4. Cleaning numerical columns (removing currency symbols from sales, dealing with negative values).
5. Formatting postal codes (adding back leading zeros).
6. Imputing missing values with statistical methods (median imputation for sales, mode for quantities).

In [1]:
import pandas as pd
import numpy as np

# Load messy transactions
df = pd.read_csv('dirty_store_transactions.csv')
print(f"Initial shape: {df.shape}")
df.head(10)

Initial shape: (1025, 10)


,Row ID,Order ID,Order Date,Customer Name,Segment,Country,Postal Code,Category,Sales,Quantity
0,528,CA-2018-118168,05/13/2016,Darrin Van Huff,corporate,USA,77041.0,Technology,629.86,4.0
1,360,CA-2016-106291,2017-09-15,Ken Lonsdale,Consumer,US,77041.0,Office Supplies,1464.64,5.0
2,448,CA-2017-133460,11/29/2017,Darrin Van Huff,Home Office,United States,10008.0,Office Supplies,184.82,9.0
3,32,CA-2019-119476,03-Jul-18,Ken Lonsdale,Consumer,NaN,77041.0,Technology,$745.52,9.0
4,622,CA-2018-106179,05-Mar-16,Ruben Hope,Home Office,United States,10008.0,Technology,1097.8,8.0
5,591,CA-2017-103836,2017-10-29,Brosina Hoffman,Consumer,United States,10008.0,Office Supplies,370.56,5.0
6,906,CA-2018-166411,02/10/2017,Ruben Hope,Home Office,United States of America,2108.0,Technology,711.78 USD,9.0
7,738,CA-2016-116749,2019-02-24,CLAIRE GUTE,Consumer,United States of America,90036.0,office supplies,1104.47,5.0
8,77,CA-2019-172856,2018-10-12,Ken Lonsdale,Corporate,United States of America,10008.0,Technology,627.68,2.0
9,949,CA-2018-194443,03-Aug-18,Ken Lonsdale,home office,USA,NaN,Technology,$417.96,NaN


### 1. Identify Missing Values and Data Types
Let's inspect the column types and count missing values in each column.

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         1025 non-null   int64  
 1   Order ID       1025 non-null   object 
 2   Order Date     1001 non-null   object 
 3   Customer Name  971 non-null    object 
 4   Segment        987 non-null    object 
 5   Country        974 non-null    object 
 6   Postal Code    986 non-null    float64
 7   Category       1025 non-null   object 
 8   Sales          992 non-null    object 
 9   Quantity       984 non-null    float64
dtypes: float64(2), int64(1), object(7)
memory usage: 80.2+ KB


In [3]:
print("Missing values per column:")
df.isnull().sum()

Missing values per column:


Row ID            0
Order ID          0
Order Date       24
Customer Name    54
Segment          38
Country          51
Postal Code      39
Category          0
Sales            33
Quantity         41
dtype: int64

### 2. Handle Duplicate Records
Let's check for duplicate records. We want to check for exact duplicates first, then logical duplicates (where the same transaction details are logged, possibly under a different Row ID).

In [4]:
# Check for exact duplicate rows
exact_dupes = df.duplicated()
print(f"Exact duplicates found: {exact_dupes.sum()}")

# Check duplicates ignoring 'Row ID'
logical_dupes = df.duplicated(subset=[col for col in df.columns if col != 'Row ID'])
print(f"Logical duplicates (ignoring Row ID): {logical_dupes.sum()}")

Exact duplicates found: 15
Logical duplicates (ignoring Row ID): 25


In [5]:
# Let's drop exact duplicates first
df = df.drop_duplicates()

# Now drop logical duplicates, keeping the first entry
df = df.drop_duplicates(subset=[col for col in df.columns if col != 'Row ID'], keep='first')
print(f"Shape after removing duplicates: {df.shape}")

Shape after removing duplicates: (1000, 10)


### 3. Clean Text Columns
Text fields like Customer Name, Segment, Category, and Country have casing inconsistencies and typos. Let's inspect them.

In [6]:
print("Unique segments:", df['Segment'].unique())
print("Unique categories:", df['Category'].unique())
print("Unique countries:", df['Country'].unique())

Unique segments: ['corporate' 'Consumer' 'Home Office' 'Corporate' 'home office' nan
 'consumer' 'Consumer_typo']
Unique categories: ['Technology' 'Office Supplies' 'office supplies' 'Furniture' 'furniture'
 'technology']
Unique countries: ['USA' 'US' 'United States' nan 'United States of America']


In [7]:
# Customer Name: strip whitespace, title case, and fill missing with 'Unknown'
df['Customer Name'] = df['Customer Name'].str.strip().str.title()
df['Customer Name'] = df['Customer Name'].fillna('Unknown')

# Category: strip whitespace, title case
df['Category'] = df['Category'].str.strip().str.title()

# Segment: clean casing and specific typo 'consumer_typo'
df['Segment'] = df['Segment'].str.strip().str.title()
df['Segment'] = df['Segment'].replace('Consumer_Typo', 'Consumer')
df['Segment'] = df['Segment'].fillna('Consumer') # fill missing with majority class

# Country: standardize to 'United States'
country_map = {
    'US': 'United States',
    'USA': 'United States',
    'United States Of America': 'United States',
    'United States': 'United States'
}
df['Country'] = df['Country'].str.strip().str.title()
df['Country'] = df['Country'].map(country_map).fillna('United States')

# Verify changes
print("Cleaned segments:", df['Segment'].unique())
print("Cleaned categories:", df['Category'].unique())
print("Cleaned countries:", df['Country'].unique())

Cleaned segments: ['Corporate' 'Consumer' 'Home Office']
Cleaned categories: ['Technology' 'Office Supplies' 'Furniture']
Cleaned countries: ['United States']


### 4. Parse Dates
Dates are stored in different formats and some are invalid. Let's parse them using pandas `to_datetime` with `errors='coerce'` to flag invalid dates as NaNs.

In [8]:
# Parse dates using format='mixed' and coerce to handle invalid dates
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', errors='coerce')
print("Missing dates after parsing:", df['Order Date'].isnull().sum())

Missing dates after parsing: 46


In [9]:
# Show rows where dates are null (the invalid/missing dates)
df[df['Order Date'].isnull()]

,Row ID,Order ID,Order Date,Customer Name,Segment,Country,Postal Code,Category,Sales,Quantity
21,823,CA-2017-169374,NaT,Brosina Hoffman,Consumer,United States,60610.0,Technology,264.17,9.0
83,353,CA-2018-110874,NaT,Darrin Van Huff,Consumer,United States,2108.0,Office Supplies,"$1,439.03",7.0
87,595,CA-2017-195414,NaT,Sean O'Donnell,Home Office,United States,77041.0,Furniture,866.94,10.0
91,791,CA-2017-121729,NaT,Ken Lonsdale,Home Office,United States,60610.0,Technology,1427.27,7.0
121,495,CA-2016-100035,NaT,Zuschuss Donatelli,Corporate,United States,10008.0,Technology,1156.81,4.0
187,779,CA-2019-185172,NaT,Brosina Hoffman,Corporate,United States,10008.0,Furniture,968.7 USD,7.0
207,535,CA-2016-109991,NaT,Brosina Hoffman,Corporate,United States,60610.0,Furniture,$142.59,4.0
228,166,CA-2016-156418,NaT,Ruben Hope,Corporate,United States,10008.0,Technology,1028.54,2.0
265,329,CA-2019-152540,NaT,Unknown,Home Office,United States,NaN,Furniture,961.22,1.0
276,299,CA-2019-199657,NaT,Ken Lonsdale,Corporate,United States,2108.0,Furniture,297.6 USD,9.0


In [10]:
# Since order date is critical, drop rows where date is missing
df = df.dropna(subset=['Order Date'])
print(f"Shape after dropping missing dates: {df.shape}")

Shape after dropping missing dates: (954, 10)


### 5. Clean Sales Column
Sales is currently a string because of currency symbols (`$`), commas, and units like `"USD"`. It also contains negative outliers like `-999`. Let's clean and convert it to numeric.

In [11]:
# Remove currency symbols, commas, spaces and text
df['Sales'] = df['Sales'].astype(str)
df['Sales'] = df['Sales'].str.replace('$', '', regex=False)
df['Sales'] = df['Sales'].str.replace(',', '', regex=False)
df['Sales'] = df['Sales'].str.replace(' USD', '', regex=False)
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')

# Check distribution of sales
df['Sales'].describe()

count     925.000000
mean      700.749070
std       499.495683
min      -999.000000
25%       370.560000
50%       712.410000
75%      1104.470000
max      1498.970000
Name: Sales, dtype: float64

We have some negative sales (like -999.0) and missing sales values. Let's set negative values to NaN and impute all NaNs using the median sales value of their product category.

In [12]:
# Treat negative sales as missing values
df.loc[df['Sales'] < 0, 'Sales'] = np.nan

# Impute missing sales with the median sales of their product category
df['Sales'] = df.groupby('Category')['Sales'].transform(lambda x: x.fillna(x.median()))
print("Missing sales remaining:", df['Sales'].isnull().sum())
df['Sales'].describe()

Missing sales remaining: 0


count     954.000000
mean      742.932600
std       412.877692
min         5.430000
25%       417.967500
50%       735.135000
75%      1088.062500
max      1498.970000
Name: Sales, dtype: float64

### 6. Clean Quantity Column
Let's clean and convert the quantity column to integer, filling missing values with the median quantity (which is typically a safe default for count data).

In [13]:
# Convert to numeric first
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

# Fill missing quantities with median
median_qty = df['Quantity'].median()
df['Quantity'] = df['Quantity'].fillna(median_qty).astype(int)

print("Quantity summary:")
df['Quantity'].value_counts()

Quantity summary:


Quantity
5     126
1     105
6     103
4     100
7      99
10     93
3      87
2      86
9      84
8      71
Name: count, dtype: int64

### 7. Standardize Postal Codes
Postal codes are read as floats or strings of varying lengths. Let's convert them to 5-digit strings, prefixing leading zeros if they were lost.

In [14]:
def clean_postal_code(val):
    if pd.isnull(val):
        return 'Unknown'
    # Convert float representation to string, stripping decimals
    val_str = str(val).split('.')[0].strip()
    if val_str == 'nan' or val_str == '':
        return 'Unknown'
    # Pad with leading zeros up to 5 characters
    return val_str.zfill(5)

df['Postal Code'] = df['Postal Code'].apply(clean_postal_code)
print("Unique postal codes preview:")
df['Postal Code'].value_counts().head(10)

Unique postal codes preview:


Postal Code
02108      294
90036      164
60610      156
10008      156
77041      146
Unknown     38
Name: count, dtype: int64

### 8. Final Inspection & Export
Let's view the clean dataset info, make sure there are no nulls left, and save it to a CSV file.

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 954 entries, 0 to 1024
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         954 non-null    int64         
 1   Order ID       954 non-null    object        
 2   Order Date     954 non-null    datetime64[ns]
 3   Customer Name  954 non-null    object        
 4   Segment        954 non-null    object        
 5   Country        954 non-null    object        
 6   Postal Code    954 non-null    object        
 7   Category       954 non-null    object        
 8   Sales          954 non-null    float64       
 9   Quantity       954 non-null    int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(6)
memory usage: 82.0+ KB


In [16]:
df.head(10)

,Row ID,Order ID,Order Date,Customer Name,Segment,Country,Postal Code,Category,Sales,Quantity
0,528,CA-2018-118168,2016-05-13,Darrin Van Huff,Corporate,United States,77041,Technology,629.86,4
1,360,CA-2016-106291,2017-09-15,Ken Lonsdale,Consumer,United States,77041,Office Supplies,1464.64,5
2,448,CA-2017-133460,2017-11-29,Darrin Van Huff,Home Office,United States,10008,Office Supplies,184.82,9
3,32,CA-2019-119476,2018-07-03,Ken Lonsdale,Consumer,United States,77041,Technology,745.52,9
4,622,CA-2018-106179,2016-03-05,Ruben Hope,Home Office,United States,10008,Technology,1097.80,8
5,591,CA-2017-103836,2017-10-29,Brosina Hoffman,Consumer,United States,10008,Office Supplies,370.56,5
6,906,CA-2018-166411,2017-02-10,Ruben Hope,Home Office,United States,02108,Technology,711.78,9
7,738,CA-2016-116749,2019-02-24,Claire Gute,Consumer,United States,90036,Office Supplies,1104.47,5
8,77,CA-2019-172856,2018-10-12,Ken Lonsdale,Corporate,United States,10008,Technology,627.68,2
9,949,CA-2018-194443,2018-08-03,Ken Lonsdale,Home Office,United States,Unknown,Technology,417.96,5


In [17]:
# Save clean dataset
df.to_csv('cleaned_store_transactions.csv', index=False)
print("Cleaned dataset successfully exported!")

Cleaned dataset successfully exported!
